In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: Read the dataset Q1_data.csv using read_csv()

csv_path = os.path.join(path, "Q1_data.csv")

data = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()

data.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()

data.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()

data.describe()

In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data
df = data.drop("Order_ID", axis=1)
df

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately
# (Hint: I guess you want to have a closer look at the
# columns with missing values :) )

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


In [ ]:
df = df.dropna()
check_missing_values(df)
df.info()

In [ ]:
# Task 3: Write your code here: Check and remove duplicates if any exist

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df = df.drop_duplicates(inplace=False)
check_duplicates(df)

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df


In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)


In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']
y

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_mse = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model = RandomForestRegressor(n_estimators=200)
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mse = sklearn_mse(y_test, y_pred)

  # Store results
  lr_mse.append(mse)

print(f"  MSE:  {np.mean(lr_mse)}")


In [ ]:
# Task 1: Write your code here:
from sklearn.linear_model import Ridge, Lasso

models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
}
coeffs = {"Lasso": Lasso(alpha=1.0,  max_iter=10000), "Ridge" :Ridge(alpha=1.0, max_iter=10000)  }


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: